In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
env_file = next((p / "env.env" for p in [ROOT, *ROOT.parents] if (p / "env.env").exists()), None)
if env_file:
    load_dotenv(env_file)

In [ ]:
import torch
from torch.nn import functional as F

from adaloop import api

In [6]:
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i))

CUDA: True
GPU count: 8
NVIDIA A40
NVIDIA A40
NVIDIA A40
NVIDIA A40
NVIDIA A40
NVIDIA A40
NVIDIA A40
NVIDIA A40


In [7]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
load_dotenv((project_root /'env.env').resolve())
data_dir = project_root / "datasets"

Project Root: /coc/scratch/vchopra/learning_env/Deep-Learning


### Step-1: - Setup Model 

In [ ]:
cfg = api.load_config(ROOT / "configs/mvp.yaml")
vlm = api.load_model(cfg)
model, processor = vlm.model, vlm.processor
language_model = model.model.language_model
print("Device map:", model.hf_device_map)

In [14]:
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i)/1024**3:.2f} GiB")

GPU 0: 24.30 GiB
GPU 1: 24.91 GiB
GPU 2: 0.00 GiB
GPU 3: 0.00 GiB
GPU 4: 0.00 GiB
GPU 5: 0.00 GiB
GPU 6: 0.00 GiB
GPU 7: 0.00 GiB


In [ ]:
from collections import Counter

print("Missing keys:", vlm.loading_info.get("missing_keys"))
print("Unexpected keys:", vlm.loading_info.get("unexpected_keys"))
print("Mismatched keys:", vlm.loading_info.get("mismatched_keys"))

dtypes = Counter()
for p in model.parameters():
    dtypes[str(p.dtype)] += p.numel()
for dtype, count in dtypes.items():
    print(dtype, f"{count / 1e9:.3f}B parameters")

print("Model footprint:", model.get_memory_footprint() / 1024**3, "GiB")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.memory_allocated(i) / 1024**3, "GiB")

In [16]:
print(model.hf_device_map)

assert all(str(d) not in ("cpu", "disk", "meta") for d in model.hf_device_map.values())

{'model.language_model.embed_tokens': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 0, 'model.language_model.layers.12': 1, 'model.language_model.layers.13': 1, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 1, 'model.language_model.layers.23': 1, 'model.language_model.norm': 1, 'model.language_model.final_input': 1, 'model.lan

#### Step-2: - Test the inference 

In [ ]:
import time

def synchronize():
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)

inputs = api.prepare_inputs(vlm, "What is 12 multiplied by 5?")
print("Input tokens:", inputs["input_ids"].shape[-1])

def benchmark(inputs, n=1):
    synchronize()
    start = time.perf_counter()
    result = api.generate(vlm, inputs, max_new_tokens=n)
    synchronize()
    print("Time:", round(time.perf_counter() - start, 3), "seconds")
    print("Generated tokens:", len(result.token_ids))
    print("Output:", processor.decode(result.token_ids))

In [18]:
cfg = model.config.text_config

print("Experts:", cfg._experts_implementation)
print("Attention:", cfg._attn_implementation)
print("KV cache:", model.generation_config.use_cache)
print("Device map:", model.hf_device_map)

from importlib.util import find_spec
print("Causal Conv1D:", find_spec("causal_conv1d") is not None)

Experts: grouped_mm
Attention: eager
KV cache: None
Device map: {'model.language_model.embed_tokens': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 0, 'model.language_model.layers.12': 1, 'model.language_model.layers.13': 1, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 1, 'model.language_model.layers.23': 1, 'model.language_m

In [ ]:
benchmark(inputs, 1)
benchmark(inputs, 8)

In [ ]:
# Text inference
print(api.generate(vlm, inputs, cfg["max_new_tokens"]).text)

In [ ]:
# Image inference
# image = api.load_image("test.jpg", cfg["res_px"])
# print(api.generate(vlm, api.prepare_inputs(vlm, "What animal is in this image?", image), 128).text)

In [ ]:
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
question = "What animal is on the candy?"

In [ ]:
image = api.load_image(url, max_side=4032)  # full resolution
inputs = api.prepare_inputs(vlm, question, image)

print("Image resolution:", image.size)
print("Total tokens:", inputs["input_ids"].shape[-1])
print("Image grid:", inputs.get("image_grid_thw"))
print("Visual tokens:", (inputs["input_ids"] == model.config.image_token_id).sum().item())

In [ ]:
image = api.load_image(url, cfg["res_px"])
inputs = api.prepare_inputs(vlm, question, image)
print("Resolution:", image.size)
print("Total tokens:", inputs["input_ids"].shape[-1])
display(image)

In [ ]:
print(api.generate(vlm, inputs, cfg["max_new_tokens"]).text)

### Step-3: - Variable number of Loops

In [ ]:
language_model

In [ ]:
for R in range(1, 6):
    api.set_loops(vlm, R)
    print(R, language_model.config.loop_num_passes)

In [ ]:
def test_loop_execution(R):
    api.set_loops(vlm, R)
    actual = api.count_layer_calls(vlm, inputs)
    expected = api.expected_layer_calls(vlm)
    print(f"R = {R}")
    print("Expected:", expected)
    print("Actual:  ", actual)
    print("Total executions:", sum(actual))
    print("Passed:", actual == expected)

for R in range(1, 6):
    test_loop_execution(R)

In [ ]:
inputs = api.prepare_inputs(vlm, question, image)

In [ ]:
@torch.inference_mode()
def get_generation_logits(R):
    # needs per-step logits, which api.generate does not return
    api.set_loops(vlm, R)
    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        use_cache=True,
        return_dict_in_generate=True,
        output_logits=True,
    )
    tokens = out.sequences[0, inputs["input_ids"].shape[-1]:]
    return {
        "tokens": tokens.cpu(),
        "logits": torch.stack(out.logits).squeeze(1).float().cpu(),
        "answer": processor.decode(tokens, skip_special_tokens=True),
    }

In [ ]:
generations = {R: get_generation_logits(R) for R in range(1, 6)}

for R, result in generations.items():
    print(f"R={R}: {result['answer']}")

R=1: We need answer image question. Need inspect image. User asks what animal is on the candy. Need identify symbol on candies. Let's zoom mentally: image shows four round candies/buttons? Two teal/green on left, orange and green on right. On orange candy there is black symbol resembling a treble clef? Wait question asks animal. Maybe the black mark on candies is a snail? Let's examine.

The
R=2: We need answer image question. Need inspect image. User asks what animal is on the candy. Need identify logo/design on candies. Let's zoom mentally. Image shows four candy-like oval candies in palm. Two teal/green candies with black design, two orange/green. The black design on orange candy looks like a treble clef? Wait question asks animal. Maybe the logo is a frog? Let's examine.
R=3: We need answer image question. Need inspect image. User asks what animal is on the candy. Need identify logo/design on candies. Let's zoom mentally: image shows four small round candies/buttons? Teal candies h

In [41]:
for i, token in enumerate(generations[1]["tokens"]):
    print(i, repr(processor.decode([token.item()])))

0 '<|message_model|>'
1 '<|content_thinking|>'
2 'We'
3 ' need'
4 ' answer'
5 ' image'
6 ' question'
7 '.'
8 ' Need'
9 ' inspect'
10 ' image'
11 '.'
12 ' User'
13 ' asks'
14 ' what'
15 ' animal'
16 ' is'
17 ' on'
18 ' the'
19 ' '
20 'candy'
21 '.'
22 ' Need'
23 ' identify'
24 ' symbol'
25 ' on'
26 ' cand'
27 'ies'
28 '.'
29 ' Let'
30 "'"
31 's'
32 ' zoom'
33 ' ment'
34 'ally'
35 ':'
36 ' image'
37 ' shows'
38 ' four'
39 ' round'
40 ' cand'
41 'ies'
42 '/'
43 'buttons'
44 '?'
45 ' Two'
46 ' '
47 'teal'
48 '/'
49 'green'
50 ' on'
51 ' left'
52 ','
53 ' orange'
54 ' and'
55 ' green'
56 ' on'
57 ' right'
58 '.'
59 ' On'
60 ' orange'
61 ' '
62 'candy'
63 ' there'
64 ' is'
65 ' black'
66 ' symbol'
67 ' '
68 'resembl'
69 'ing'
70 ' a'
71 ' '
72 'treble'
73 ' '
74 'clef'
75 '?'
76 ' Wait'
77 ' question'
78 ' asks'
79 ' animal'
80 '.'
81 ' Maybe'
82 ' the'
83 ' black'
84 ' mark'
85 ' on'
86 ' cand'
87 'ies'
88 ' is'
89 ' a'
90 ' '
91 'snail'
92 '?'
93 ' Let'
94 "'"
95 's'
96 ' examine'
97 '.'
9

### Compare Output Distibutions 

In [42]:
t = 12

reference = generations[1]["logits"][t].double()

for R, result in generations.items():
    assert torch.equal(
        result["tokens"][:t],
        generations[1]["tokens"][:t]
    )

    z = result["logits"][t].double()

    kl = F.kl_div(
        z.log_softmax(-1),
        reference.softmax(-1),
        reduction="sum"
    ).item()

    print(f"R={R}, KL={kl:.6f}, Token={processor.decode([z.argmax().item()])}")

R=1, KL=0.000000, Token= User
R=2, KL=0.010508, Token= User
R=3, KL=0.028745, Token= User
R=4, KL=0.036395, Token= It
R=5, KL=0.031374, Token= It
